# CDPL PAMAP2 Statistical Benchmark Analysis

## Imports and setup

Core imports used by the training, evaluation, and benchmark utilities.

In [ ]:
from __future__ import annotations

import copy
import math
import os
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## Reproducibility

Sets seeds and PyTorch backend settings for repeatable runs.

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

## Configuration

Defines the main CDPL/PAMAP2 configuration, including data paths, windowing, model size, training, and personalization settings.

In [ ]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    # ----- data -----
    data_dir: str = "/content/PAMAP2_Dataset/Protocol"
    subject_glob: str = "subject*.dat"
    valid_activity_ids: Tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 12, 13, 16, 17, 24)

    seq_len: int = 100
    stride: int = 50
    drop_mixed_windows: bool = True
    purge_gap_raw: int = 50  # active gap between sub-splits inside the same constant-label segment

    train_subject_train_ratio: float = 0.85
    train_subject_val_ratio: float = 0.15

    heldout_support_ratio: float = 0.20
    heldout_adapt_val_ratio: float = 0.20
    heldout_test_ratio: float = 0.60

    min_windows_per_partition: int = 8

    # ----- optimization -----
    batch_size: int = 128
    personal_batch_size: int = 64
    num_workers: int = 0
    pin_memory: bool = True

    rounds: int = 15
    local_epochs: int = 6
    personalization_epochs: int = 10

    lr_encoder: float = 3e-4
    lr_personal: float = 1e-2
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    lambda_align: float = 0.7
    lambda_A: float = 1e-3
    lambda_B: float = 5e-3

    server_geom_steps: int = 20
    server_geom_lr: float = 2e-2
    server_alt_iters: int = 3

    # ----- model -----
    d_model: int = 192
    emb_dim: int = 192
    n_heads: int = 6
    n_layers: int = 3
    ff_dim: int = 384
    dropout: float = 0.2
    proto_rank: int = 8

    # ----- misc -----
    ece_bins: int = 15
    seed: int = 42
    amp: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./ccd_pamap2_runs"

    # ----- stabilization / scoring -----
    class_weight_power: float = 0.5
    class_weight_max: float = 4.0
    min_proto_samples_per_class: int = 4

    tau_init: float = 8.0
    tau_min: float = 1.5
    tau_max: float = 20.0
    lambda_tau: float = 1e-4   # reduced from 5e-4

    lambda_personal_anchor: float = 3e-4   # reduced from 2e-3

    server_proto_momentum: float = 0.85
    server_basis_momentum: float = 0.80

    client_val_weight_power: float = 0.5
    round_score_f1_w: float = 0.70
    round_score_acc_w: float = 0.30
    round_score_ece_w: float = 0.05

    # ----- adaptive personalization -----
    personal_tau_lr_mult: float = 0.25
    personalization_epochs_min: int = 6
    personalization_epochs_max: int = 14
    personalization_patience: int = 3
    hard_subject_f1_threshold: float = 0.55
    easy_subject_f1_threshold: float = 0.78

    # ----- personalization gating -----
    personalization_gate_score_margin: float = 0.002
    personalization_gate_score_margin_easy: float = 0.010
    personalization_gate_f1_margin: float = 0.000

    # for faster debugging
    run_single_heldout: Optional[str] = None  # e.g. "subject101"

    def __post_init__(self) -> None:
        assert abs(self.train_subject_train_ratio + self.train_subject_val_ratio - 1.0) < 1e-8
        assert abs(
            self.heldout_support_ratio + self.heldout_adapt_val_ratio + self.heldout_test_ratio - 1.0
        ) < 1e-8
        os.makedirs(self.save_dir, exist_ok=True)

## PAMAP2 columns

Defines the PAMAP2 sensor column names and selects the feature columns used by the model.

In [ ]:
# ============================================================
# PAMAP2 columns
# ============================================================

def pamap2_columns() -> List[str]:
    return [
        "timestamp",
        "activity_id",
        "heart_rate",
        "hand_temperature",
        "hand_acc16_x",
        "hand_acc16_y",
        "hand_acc16_z",
        "hand_acc6_x",
        "hand_acc6_y",
        "hand_acc6_z",
        "hand_gyro_x",
        "hand_gyro_y",
        "hand_gyro_z",
        "hand_mag_x",
        "hand_mag_y",
        "hand_mag_z",
        "hand_orient_1",
        "hand_orient_2",
        "hand_orient_3",
        "hand_orient_4",
        "chest_temperature",
        "chest_acc16_x",
        "chest_acc16_y",
        "chest_acc16_z",
        "chest_acc6_x",
        "chest_acc6_y",
        "chest_acc6_z",
        "chest_gyro_x",
        "chest_gyro_y",
        "chest_gyro_z",
        "chest_mag_x",
        "chest_mag_y",
        "chest_mag_z",
        "chest_orient_1",
        "chest_orient_2",
        "chest_orient_3",
        "chest_orient_4",
        "ankle_temperature",
        "ankle_acc16_x",
        "ankle_acc16_y",
        "ankle_acc16_z",
        "ankle_acc6_x",
        "ankle_acc6_y",
        "ankle_acc6_z",
        "ankle_gyro_x",
        "ankle_gyro_y",
        "ankle_gyro_z",
        "ankle_mag_x",
        "ankle_mag_y",
        "ankle_mag_z",
        "ankle_orient_1",
        "ankle_orient_2",
        "ankle_orient_3",
        "ankle_orient_4",
    ]


def feature_columns() -> List[str]:
    cols = pamap2_columns()
    return [c for c in cols if c not in ("timestamp", "activity_id")]

## Metrics

Implements accuracy, macro-F1, expected calibration error, Brier score, and selection-score utilities.

In [ ]:
# ============================================================
# Metrics
# ============================================================

def expected_calibration_error(
    probs: np.ndarray,
    labels: np.ndarray,
    n_bins: int = 15,
) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        if mask.any():
            acc_bin = accuracies[mask].mean()
            conf_bin = confidences[mask].mean()
            ece += float(mask.mean()) * abs(float(acc_bin) - float(conf_bin))
    return float(ece)


def multiclass_brier_score(probs: np.ndarray, labels: np.ndarray, num_classes: int) -> float:
    one_hot = np.eye(num_classes, dtype=np.float32)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def summarize_probs(
    probs: np.ndarray,
    labels: np.ndarray,
    num_classes: int,
    ece_bins: int,
) -> Dict[str, float]:
    preds = probs.argmax(axis=1)
    return {
        "acc": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "ece": expected_calibration_error(probs, labels, n_bins=ece_bins),
        "brier": multiclass_brier_score(probs, labels, num_classes=num_classes),
    }

def weighted_mean(values: List[float], weights: List[float]) -> float:
    if len(values) == 0:
        return 0.0
    w = np.asarray(weights, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    w = w / max(w.sum(), 1e-12)
    return float(np.sum(w * v))


def aggregate_round_metrics(
    per_client_metrics: Dict[str, Dict[str, float]],
    per_client_val_sizes: Dict[str, int],
    cfg: Config,
) -> Tuple[float, float, float, float]:
    client_ids = sorted(per_client_metrics.keys())
    weights = [max(1, per_client_val_sizes[sid]) ** cfg.client_val_weight_power for sid in client_ids]

    mean_f1 = weighted_mean([per_client_metrics[sid]["f1"] for sid in client_ids], weights)
    mean_acc = weighted_mean([per_client_metrics[sid]["acc"] for sid in client_ids], weights)
    mean_ece = weighted_mean([per_client_metrics[sid]["ece"] for sid in client_ids], weights)

    score = (
        cfg.round_score_f1_w * mean_f1
        + cfg.round_score_acc_w * mean_acc
        - cfg.round_score_ece_w * mean_ece
    )
    return mean_f1, mean_acc, mean_ece, score

def choose_personalization_hparams(
    init_f1: float,
    cfg: Config,
) -> Dict[str, float]:
    """
    Hard subjects get more epochs and weaker anchoring.
    Easy subjects get fewer epochs and slightly stronger anchoring.
    """
    if init_f1 < cfg.hard_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_max,
            "anchor_w": cfg.lambda_personal_anchor * 0.35,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.75,
            "warm_epochs": 2,
        }
    elif init_f1 > cfg.easy_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_min,
            "anchor_w": cfg.lambda_personal_anchor * 2.0,
            "lr_A": cfg.lr_personal * 0.85,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.50,
            "warm_epochs": 1,
        }
    else:
        return {
            "epochs": int(round(0.5 * (cfg.personalization_epochs_min + cfg.personalization_epochs_max))),
            "anchor_w": cfg.lambda_personal_anchor,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult,
            "warm_epochs": 2,
        }

def personalization_selection_score(metrics: Dict[str, float]) -> float:
    return 0.90 * metrics["f1"] + 0.10 * metrics["acc"] - 0.05 * metrics["ece"]

## Data structures

Small dataset and dataclass containers used to represent subject partitions and LOSO fold data.

In [ ]:
# ============================================================
# Data structures
# ============================================================

class WindowDataset(Dataset):
    def __init__(self, windows: np.ndarray, labels: np.ndarray):
        assert len(windows) == len(labels)
        self.x = torch.from_numpy(np.ascontiguousarray(windows)).float()
        self.y = torch.from_numpy(np.ascontiguousarray(labels)).long()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


@dataclass
class SubjectPartition:
    windows: np.ndarray
    labels: np.ndarray


@dataclass
class FoldData:
    heldout_id: str
    train_ids: List[str]
    train_subjects: Dict[str, Dict[str, SubjectPartition]]
    heldout_subject: Dict[str, SubjectPartition]
    num_classes: int
    feature_dim: int
    label_map: Dict[int, int]

## Loading and preprocessing

Loads PAMAP2 files, creates leakage-safe constant-label windows, splits subjects, and standardizes features.

In [ ]:
# ============================================================
# Loading and preprocessing
# ============================================================

def discover_subject_files(cfg: Config) -> Dict[str, Path]:
    paths = sorted(Path(cfg.data_dir).glob(cfg.subject_glob))
    if not paths:
        raise FileNotFoundError(
            f"No PAMAP2 files found in {cfg.data_dir!r} with glob {cfg.subject_glob!r}."
        )
    return {p.stem: p for p in paths}


def load_subject_dataframe(file_path: Path, cfg: Config) -> pd.DataFrame:
    cols = pamap2_columns()
    df = pd.read_csv(file_path, sep=r"\s+", header=None, names=cols, engine="python")

    # only keep valid activities
    df = df[df["activity_id"].isin(cfg.valid_activity_ids)].copy()
    if df.empty:
        raise RuntimeError(f"{file_path.name} has no rows for valid PAMAP2 activity IDs.")

    # PAMAP2 uses -1 as missing marker
    df.replace(-1.0, np.nan, inplace=True)

    feat_cols = feature_columns()
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both", axis=0)
    df[feat_cols] = df[feat_cols].ffill().bfill()
    df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())

    return df.reset_index(drop=True)


def build_global_label_map(subject_dfs: Dict[str, pd.DataFrame]) -> Dict[int, int]:
    activity_ids = sorted({int(a) for df in subject_dfs.values() for a in df["activity_id"].unique()})
    return {aid: idx for idx, aid in enumerate(activity_ids)}


def encode_subject(
    df: pd.DataFrame,
    label_map: Dict[int, int],
) -> Tuple[np.ndarray, np.ndarray]:
    X = df[feature_columns()].to_numpy(dtype=np.float32)
    y = df["activity_id"].map(label_map).to_numpy(dtype=np.int64)
    return X, y


def constant_label_segments(y: np.ndarray) -> List[Tuple[int, int, int]]:
    segments: List[Tuple[int, int, int]] = []
    if len(y) == 0:
        return segments

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1
        segments.append((s, e, int(y[s])))
        s = e
    return segments


def generate_constant_label_windows(
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
    stride: int,
    drop_mixed_windows: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    windows: List[np.ndarray] = []
    labels: List[int] = []

    if len(X) == 0:
        feat_dim = X.shape[1] if X.ndim == 2 else 0
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1

        seg_x = X[s:e]
        seg_y = y[s]
        seg_len = len(seg_x)

        if seg_len >= seq_len:
            for start in range(0, seg_len - seq_len + 1, stride):
                win_x = seg_x[start : start + seq_len]
                if drop_mixed_windows and not np.all(y[s + start : s + start + seq_len] == seg_y):
                    continue
                windows.append(win_x)
                labels.append(int(seg_y))
        s = e

    feat_dim = X.shape[1]
    if not windows:
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.stack(windows).astype(np.float32), np.array(labels, dtype=np.int64)


def empty_partition(seq_len: int, feat_dim: int) -> SubjectPartition:
    return SubjectPartition(
        windows=np.empty((0, seq_len, feat_dim), dtype=np.float32),
        labels=np.empty((0,), dtype=np.int64),
    )


def concat_partitions(parts: List[SubjectPartition], seq_len: int, feat_dim: int) -> SubjectPartition:
    non_empty = [p for p in parts if len(p.labels) > 0]
    if not non_empty:
        return empty_partition(seq_len, feat_dim)
    return SubjectPartition(
        windows=np.concatenate([p.windows for p in non_empty], axis=0).astype(np.float32),
        labels=np.concatenate([p.labels for p in non_empty], axis=0).astype(np.int64),
    )


def labels_present(labels: np.ndarray) -> List[int]:
    if len(labels) == 0:
        return []
    return np.where(np.bincount(labels) > 0)[0].tolist()


def label_histogram(labels: np.ndarray, num_classes: int) -> Dict[int, int]:
    if len(labels) == 0:
        return {}
    counts = np.bincount(labels, minlength=num_classes)
    return {i: int(c) for i, c in enumerate(counts) if c > 0}


def allocate_lengths_with_minimum(
    total_len: int,
    ratios: List[float],
    min_lengths: List[int],
) -> Optional[List[int]]:
    """
    Split total_len into k parts:
    - each part >= min_lengths[i]
    - remaining budget allocated by ratios
    """
    min_sum = int(sum(min_lengths))
    if total_len < min_sum:
        return None

    ratios_arr = np.array(ratios, dtype=np.float64)
    if ratios_arr.sum() <= 0:
        ratios_arr = np.ones_like(ratios_arr)
    ratios_arr = ratios_arr / ratios_arr.sum()

    extra_total = int(total_len - min_sum)
    raw_extra = ratios_arr * extra_total
    extra = np.floor(raw_extra).astype(np.int64)

    remainder = extra_total - int(extra.sum())
    if remainder > 0:
        frac = raw_extra - extra
        order = np.argsort(-frac)
        for idx in order[:remainder]:
            extra[idx] += 1

    out = (np.array(min_lengths, dtype=np.int64) + extra).astype(np.int64)
    return [int(v) for v in out]


def split_single_segment_train_val(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into train/val subranges with a purge gap,
    so train and val both contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for train + gap + val
    if seg_len < (2 * min_win + gap):
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    usable = seg_len - gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.train_subject_train_ratio, cfg.train_subject_val_ratio],
        min_lengths=[min_win, min_win],
    )
    if lengths is None:
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    train_len, val_len = lengths
    train_range = (0, train_len)
    val_range = (train_len + gap, train_len + gap + val_len)

    return {
        "train": [train_range],
        "val": [val_range],
    }


def split_single_segment_support_adapt_test(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into support/adapt_val/test subranges with purge gaps,
    so all three held-out partitions contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for support + gap + adapt + gap + test
    if seg_len < (3 * min_win + 2 * gap):
        # fallback hierarchy: if not enough for 3-way, try 2-way, else put all in test
        if seg_len >= (2 * min_win + gap):
            usable = seg_len - gap
            lengths = allocate_lengths_with_minimum(
                total_len=usable,
                ratios=[cfg.heldout_support_ratio + cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
                min_lengths=[min_win, min_win],
            )
            if lengths is None:
                return {
                    "support": [],
                    "adapt_val": [],
                    "test": [(0, seg_len)] if seg_len >= min_win else [],
                }

            sa_len, test_len = lengths

            # Try splitting the first block into support + adapt_val too
            if sa_len >= (2 * min_win + gap):
                sa_usable = sa_len - gap
                sa_lengths = allocate_lengths_with_minimum(
                    total_len=sa_usable,
                    ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio],
                    min_lengths=[min_win, min_win],
                )
                if sa_lengths is not None:
                    support_len, adapt_len = sa_lengths
                    return {
                        "support": [(0, support_len)],
                        "adapt_val": [(support_len + gap, support_len + gap + adapt_len)],
                        "test": [(sa_len + gap, sa_len + gap + test_len)],
                    }

            # If still impossible, use support + test and leave adapt empty
            return {
                "support": [(0, sa_len)],
                "adapt_val": [],
                "test": [(sa_len + gap, sa_len + gap + test_len)],
            }

        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    usable = seg_len - 2 * gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
        min_lengths=[min_win, min_win, min_win],
    )
    if lengths is None:
        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    support_len, adapt_len, test_len = lengths

    support_range = (0, support_len)
    adapt_range = (support_len + gap, support_len + gap + adapt_len)
    test_range = (support_len + gap + adapt_len + gap, support_len + gap + adapt_len + gap + test_len)

    return {
        "support": [support_range],
        "adapt_val": [adapt_range],
        "test": [test_range],
    }


def windows_from_absolute_ranges(
    X: np.ndarray,
    y: np.ndarray,
    abs_ranges: List[Tuple[int, int]],
    cfg: Config,
) -> SubjectPartition:
    feat_dim = X.shape[1]
    parts: List[SubjectPartition] = []

    for a, b in abs_ranges:
        if (b - a) < cfg.seq_len:
            continue
        w, lab = generate_constant_label_windows(
            X[a:b],
            y[a:b],
            seq_len=cfg.seq_len,
            stride=cfg.stride,
            drop_mixed_windows=cfg.drop_mixed_windows,
        )
        if len(lab) > 0:
            parts.append(SubjectPartition(windows=w, labels=lab))

    return concat_partitions(parts, cfg.seq_len, feat_dim)


def build_subject_partitions_train(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    train_parts: List[SubjectPartition] = []
    val_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_train_val(seg_len, cfg)

        train_abs = [(s + a, s + b) for a, b in rel["train"]]
        val_abs = [(s + a, s + b) for a, b in rel["val"]]

        train_parts.append(windows_from_absolute_ranges(X, y, train_abs, cfg))
        val_parts.append(windows_from_absolute_ranges(X, y, val_abs, cfg))

    return {
        "train": concat_partitions(train_parts, cfg.seq_len, feat_dim),
        "val": concat_partitions(val_parts, cfg.seq_len, feat_dim),
    }


def build_subject_partitions_heldout(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    support_parts: List[SubjectPartition] = []
    adapt_parts: List[SubjectPartition] = []
    test_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_support_adapt_test(seg_len, cfg)

        support_abs = [(s + a, s + b) for a, b in rel["support"]]
        adapt_abs = [(s + a, s + b) for a, b in rel["adapt_val"]]
        test_abs = [(s + a, s + b) for a, b in rel["test"]]

        support_parts.append(windows_from_absolute_ranges(X, y, support_abs, cfg))
        adapt_parts.append(windows_from_absolute_ranges(X, y, adapt_abs, cfg))
        test_parts.append(windows_from_absolute_ranges(X, y, test_abs, cfg))

    return {
        "support": concat_partitions(support_parts, cfg.seq_len, feat_dim),
        "adapt_val": concat_partitions(adapt_parts, cfg.seq_len, feat_dim),
        "test": concat_partitions(test_parts, cfg.seq_len, feat_dim),
    }


def print_partition_debug(
    sid: str,
    packaged: Dict[str, SubjectPartition],
    num_classes: int,
    heldout: bool,
) -> None:
    if heldout:
        split_order = ["support", "adapt_val", "test"]
    else:
        split_order = ["train", "val"]

    print(f"[PARTITIONS][{sid}]")
    for split_name in split_order:
        part = packaged[split_name]
        print(
            f"  {split_name}: windows={len(part.labels)}, "
            f"classes={labels_present(part.labels)}, "
            f"hist={label_histogram(part.labels, num_classes)}"
        )

    if not heldout:
        train_set = set(labels_present(packaged["train"].labels))
        val_set = set(labels_present(packaged["val"].labels))
        missing = sorted(val_set - train_set)
        print(f"  train/val overlap ok? missing_val_in_train={missing}")
    else:
        support_set = set(labels_present(packaged["support"].labels))
        adapt_set = set(labels_present(packaged["adapt_val"].labels))
        test_set = set(labels_present(packaged["test"].labels))
        print(f"  support∩adapt classes={sorted(support_set & adapt_set)}")
        print(f"  support∩test classes={sorted(support_set & test_set)}")
        print(f"  adapt∩test classes={sorted(adapt_set & test_set)}")


def build_fold_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    heldout_id: str,
    label_map: Dict[int, int],
    cfg: Config,
) -> FoldData:
    train_ids = [sid for sid in subject_arrays.keys() if sid != heldout_id]

    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    heldout_subject: Dict[str, SubjectPartition] = {}

    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        if sid == heldout_id:
            packaged = build_subject_partitions_heldout(X, y, cfg)
            heldout_subject = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=True)
        else:
            packaged = build_subject_partitions_train(X, y, cfg)
            train_subjects[sid] = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in train_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across non-heldout subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in train_subjects.keys():
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for split_name in heldout_subject.keys():
        heldout_subject[split_name] = transform_partition(heldout_subject[split_name])

    for sid in train_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    for split_name in ("support", "adapt_val", "test"):
        if len(heldout_subject[split_name].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(
                f"Held-out {heldout_id} split {split_name} has too few windows "
                f"({len(heldout_subject[split_name].windows)})."
            )

    feat_dim = next(iter(subject_arrays.values()))[0].shape[1]
    return FoldData(
        heldout_id=heldout_id,
        train_ids=train_ids,
        train_subjects=train_subjects,
        heldout_subject=heldout_subject,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
    )


def build_fold_summary(fold: FoldData) -> Dict[str, Dict[str, int]]:
    out: Dict[str, Dict[str, int]] = {}
    for sid in fold.train_ids:
        out[sid] = {
            "train": int(len(fold.train_subjects[sid]["train"].labels)),
            "val": int(len(fold.train_subjects[sid]["val"].labels)),
        }
    out[fold.heldout_id] = {
        "support": int(len(fold.heldout_subject["support"].labels)),
        "adapt_val": int(len(fold.heldout_subject["adapt_val"].labels)),
        "test": int(len(fold.heldout_subject["test"].labels)),
    }
    return out


def make_loader(
    part: SubjectPartition,
    batch_size: int,
    shuffle: bool,
    cfg: Config,
) -> DataLoader:
    ds = WindowDataset(part.windows, part.labels)
    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.pin_memory and str(cfg.device).startswith("cuda")),
        persistent_workers=(cfg.num_workers > 0),
        drop_last=False,
    )
    return DataLoader(ds, **loader_kwargs)


def fold_to_loaders(
    fold: FoldData,
    cfg: Config,
) -> Dict[str, object]:
    loaders: Dict[str, object] = {
        "train_clients": {},
        "heldout": {},
        "num_classes": fold.num_classes,
        "feature_dim": fold.feature_dim,
        "heldout_id": fold.heldout_id,
        "train_ids": fold.train_ids,
    }

    for sid in fold.train_ids:
        loaders["train_clients"][sid] = {
            "train": make_loader(fold.train_subjects[sid]["train"], cfg.batch_size, True, cfg),
            "val": make_loader(fold.train_subjects[sid]["val"], cfg.batch_size, False, cfg),
            "n_train": len(fold.train_subjects[sid]["train"].labels),
        }

    for split_name in ("support", "adapt_val", "test"):
        bs = cfg.personal_batch_size if split_name != "test" else cfg.batch_size
        loaders["heldout"][split_name] = make_loader(
            fold.heldout_subject[split_name], bs, split_name == "support", cfg
        )

    loaders["summary"] = build_fold_summary(fold)
    return loaders

## Model

Temporal encoder with convolutional stem, sinusoidal positional encoding, Transformer layers, attentive pooling, and normalized embeddings.

In [ ]:
# ============================================================
# Model
# ============================================================

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TemporalStem(nn.Module):
    def __init__(self, in_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, d_model, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=1, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class AttentivePool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn = torch.softmax(self.score(x).squeeze(-1), dim=1)
        pooled = torch.sum(attn.unsqueeze(-1) * x, dim=1)
        return pooled


class TemporalEncoder(nn.Module):
    def __init__(self, in_dim: int, cfg: Config):
        super().__init__()
        self.stem = TemporalStem(in_dim, cfg.d_model, cfg.dropout)
        self.pos = SinusoidalPositionalEncoding(cfg.d_model, max_len=max(2048, cfg.seq_len + 8))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=False,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.pool = AttentivePool(cfg.d_model)
        self.out_norm = nn.LayerNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h = self.pos(h)
        h = self.encoder(h)
        h = self.pool(h)
        h = self.out_norm(h)
        z = self.proj(h)
        z = F.normalize(z, dim=-1)
        return z

## Prototype utilities

Prototype deformation, cosine-style logits, local loss, class weighting, and related helper functions.

In [ ]:
# ============================================================
# Prototype utilities
# ============================================================

def orthonormal_random(d: int, r: int, device: torch.device) -> torch.Tensor:
    q, _ = torch.linalg.qr(torch.randn(d, r, device=device))
    return q[:, :r].contiguous()


def personalized_prototypes_ccd(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
) -> torch.Tensor:
    proto = global_proto + client_A @ proto_basis.T
    proto = F.normalize(proto, dim=-1)
    return proto


def proto_logits(
    z: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
) -> torch.Tensor:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    tau = torch.exp(log_tau).clamp(cfg.tau_min, cfg.tau_max)
    return tau * (z @ P_i.T)


def local_loss(
    z: torch.Tensor,
    y: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    class_weight: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)

    loss_proto = F.cross_entropy(
        logits,
        y,
        weight=class_weight,
        label_smoothing=0.05,
    )
    align = 1.0 - torch.sum(z * P_i[y], dim=1).mean()
    reg_A = client_A.pow(2).mean()
    reg_tau = (log_tau - math.log(cfg.tau_init)) ** 2

    loss = (
        loss_proto
        + cfg.lambda_align * align
        + cfg.lambda_A * reg_A
        + cfg.lambda_tau * reg_tau
    )

    stats = {
        "loss_proto": float(loss_proto.detach().item()),
        "loss_align": float(align.detach().item()),
        "loss_regA": float(reg_A.detach().item()),
        "loss_regTau": float(reg_tau.detach().item()),
        "tau": float(torch.exp(log_tau.detach()).clamp(cfg.tau_min, cfg.tau_max).item()),
    }
    return loss, stats


def make_class_weight_from_loader(
    train_loader: DataLoader,
    num_classes: int,
    device: torch.device,
    cfg: Config,
) -> torch.Tensor:
    labels = train_loader.dataset.y.detach().cpu().numpy()
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)

    present = counts > 0
    weights = np.ones(num_classes, dtype=np.float32)

    if present.any():
        ref = float(np.median(counts[present]))
        weights[present] = np.power(ref / np.clip(counts[present], 1.0, None), cfg.class_weight_power)
        weights[present] = np.clip(weights[present], 1.0 / cfg.class_weight_max, cfg.class_weight_max)
        weights[present] /= max(weights[present].mean(), 1e-8)

    return torch.tensor(weights, dtype=torch.float32, device=device)

## Evaluation

Prediction and evaluation helpers for model/prototype heads.

In [ ]:
# ============================================================
# Evaluation
# ============================================================

@torch.inference_mode()
def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    if not probs_all:
        num_classes = global_proto.shape[0]
        return np.empty((0, num_classes), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Dict[str, float]:
    probs, labels = predict_loader(
        model, loader, global_proto, proto_basis, client_A, log_tau, device, cfg, cfg.amp
    )
    return summarize_probs(probs, labels, num_classes=global_proto.shape[0], ece_bins=cfg.ece_bins)

## Client training and prototype extraction

Local client training plus confidence-weighted empirical prototype extraction.

In [ ]:
# ============================================================
# Client training and prototype extraction
# ============================================================

def clone_model(model: nn.Module) -> nn.Module:
    return copy.deepcopy(model)


def train_one_client(
    global_model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    A_init: torch.Tensor,
    log_tau_init: torch.Tensor,
    cfg: Config,
    device: torch.device,
    warmup_extract: bool = False,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    del val_loader  # local validation was unused by the outer loop and only added runtime

    local_model = clone_model(global_model).to(device)

    client_A = nn.Parameter(A_init.clone().to(device))
    log_tau = nn.Parameter(log_tau_init.clone().to(device))
    class_weight = make_class_weight_from_loader(train_loader, global_proto.shape[0], device, cfg)

    optimizer = torch.optim.AdamW(
        list(local_model.parameters()) + [client_A, log_tau],
        lr=cfg.lr_encoder,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    optimizer_steps = 0
    scheduler_steps = 0

    for _epoch in range(cfg.local_epochs):
        local_model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_model(xb)
                loss, _ = local_loss(
                    z, yb, global_proto, proto_basis, client_A, log_tau, cfg, class_weight=class_weight
                )

            if amp_enabled:
                scaler.scale(loss).backward()

                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )

                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()

                if new_scale >= old_scale:
                    optimizer_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )
                optimizer.step()
                optimizer_steps += 1

        if optimizer_steps > scheduler_steps:
            scheduler.step()
            scheduler_steps += 1

    proto_summary = {}
    emp, present, counts = extract_confidence_weighted_prototypes(
        model=local_model,
        loader=train_loader,
        global_proto=global_proto,
        proto_basis=proto_basis,
        client_A=client_A.detach(),
        log_tau=log_tau.detach(),
        num_classes=global_proto.shape[0],
        emb_dim=cfg.emb_dim,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=warmup_extract,
    )
    proto_summary["empirical"] = emp
    proto_summary["present"] = present
    proto_summary["counts"] = counts

    state_dict = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}

    del local_model

    return (
        state_dict,
        client_A.detach().cpu().clone(),
        log_tau.detach().cpu().clone(),
        proto_summary,
    )


@torch.inference_mode()
def extract_confidence_weighted_prototypes(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    num_classes: int,
    emb_dim: int,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
    warmup: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    model.eval()

    proto_sum = torch.zeros(num_classes, emb_dim, device=device)
    weight_sum = torch.zeros(num_classes, device=device)
    counts = torch.zeros(num_classes, device=device)

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)

        if warmup:
            weights = torch.ones_like(yb, dtype=z.dtype, device=device)
        else:
            true_conf = probs.gather(1, yb.unsqueeze(1)).squeeze(1)
            max_conf = probs.max(dim=1).values
            weights = 0.7 * true_conf + 0.3 * max_conf

        proto_sum.index_add_(0, yb, z * weights.unsqueeze(1))
        weight_sum.index_add_(0, yb, weights)
        counts.index_add_(0, yb, torch.ones_like(weights))

    empirical = torch.zeros_like(proto_sum)
    present = (weight_sum > 0) & (counts >= cfg.min_proto_samples_per_class)

    empirical[present] = proto_sum[present] / weight_sum[present].unsqueeze(1)
    empirical[present] = F.normalize(empirical[present], dim=1)

    return empirical.detach().cpu(), present.detach().cpu(), counts.detach().cpu()

## Server aggregation and geometry update

FedAvg-style encoder aggregation and server-side prototype/basis geometry update.

In [ ]:
# ============================================================
# Server aggregation and geometry update
# ============================================================

def average_state_dicts(
    state_dicts: List[Dict[str, torch.Tensor]],
    weights: List[float],
) -> Dict[str, torch.Tensor]:
    total = float(sum(weights))
    weights = [w / total for w in weights]
    out: Dict[str, torch.Tensor] = {}
    keys = state_dicts[0].keys()
    for k in keys:
        ref = state_dicts[0][k]
        if torch.is_floating_point(ref):
            acc = None
            for sd, w in zip(state_dicts, weights):
                tensor = sd[k].float()
                acc = tensor * w if acc is None else acc + tensor * w
            out[k] = acc.to(dtype=ref.dtype)
        else:
            out[k] = ref.clone()
    return out


def solve_A_closed_form(
    empirical_proto: torch.Tensor,
    present: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    lambda_A: float,
) -> torch.Tensor:
    BtB = proto_basis.T @ proto_basis
    r = BtB.shape[0]
    inv = torch.linalg.inv(BtB + lambda_A * torch.eye(r, device=proto_basis.device, dtype=proto_basis.dtype))
    A = (empirical_proto - global_proto) @ proto_basis @ inv
    A = A * present.float().unsqueeze(1)
    return A


def server_geometry_update(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    proto_summaries: Dict[str, Dict[str, torch.Tensor]],
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Minimizes approximately:
        sum_{i,c} w_{i,c} || H_ic - p_c - B a_ic ||^2
        + lambda_A sum_i ||A_i||^2 + lambda_B ||B^T B - I||^2
    """
    old_P = global_proto.clone().to(device)
    old_B = proto_basis.clone().to(device)

    P = old_P.clone()
    B = old_B.clone()

    client_ids = list(proto_summaries.keys())
    H = torch.stack([proto_summaries[sid]["empirical"].to(device) for sid in client_ids], dim=0)
    M = torch.stack([proto_summaries[sid]["present"].to(device) for sid in client_ids], dim=0).bool()
    W = torch.stack([proto_summaries[sid]["counts"].to(device) for sid in client_ids], dim=0).float()
    W = torch.where(M, torch.sqrt(torch.clamp(W, min=0.0)), torch.zeros_like(W))

    K, C, d = H.shape
    r = B.shape[1]

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    for _ in range(cfg.server_alt_iters):
        numer = torch.zeros_like(P)
        denom = torch.zeros(C, device=device)
        for i in range(K):
            recon_free = H[i] - A[i] @ B.T
            numer += W[i].unsqueeze(1) * recon_free
            denom += W[i]

        keep = denom > 0
        P[keep] = numer[keep] / denom[keep].unsqueeze(1)
        P = F.normalize(P, dim=1)

        B_param = nn.Parameter(B.clone())
        opt = torch.optim.Adam([B_param], lr=cfg.server_geom_lr)

        for _step in range(cfg.server_geom_steps):
            opt.zero_grad(set_to_none=True)
            recon = P.unsqueeze(0) + torch.matmul(A, B_param.T)
            sq = ((H - recon) ** 2).sum(dim=2)
            data_term = (W * sq).sum() / (W.sum() + 1e-8)
            orth = ((B_param.T @ B_param) - torch.eye(r, device=device)).pow(2).mean()
            loss = data_term + cfg.lambda_B * orth
            loss.backward()
            opt.step()

        with torch.no_grad():
            q, _ = torch.linalg.qr(B_param.data)
            B = q[:, :r].contiguous()

        A_list = []
        for i in range(K):
            A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
            A_list.append(A_i)
        A = torch.stack(A_list, dim=0)

    # momentum smoothing of shared geometry
    P = F.normalize(cfg.server_proto_momentum * old_P + (1.0 - cfg.server_proto_momentum) * P, dim=1)
    B_blend = cfg.server_basis_momentum * old_B + (1.0 - cfg.server_basis_momentum) * B
    q, _ = torch.linalg.qr(B_blend)
    B = q[:, :r].contiguous()

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    client_A = {sid: A[idx].detach().cpu().clone() for idx, sid in enumerate(client_ids)}
    return P.detach(), B.detach(), client_A

## Held-out personalization

Support-set adaptation of held-out client coefficients and temperature with validation gating.

In [ ]:
# ============================================================
# Held-out personalization
# ============================================================

def adapt_heldout_client(
    model: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    cfg: Config,
    device: torch.device,
    init_log_tau: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float], Dict[str, object]]:
    frozen_model = clone_model(model).to(device)
    frozen_model.eval()
    for p in frozen_model.parameters():
        p.requires_grad = False

    C, d = global_proto.shape
    r = proto_basis.shape[1]
    global_proto = global_proto.to(device)
    proto_basis = proto_basis.to(device)

    if init_log_tau is None:
        init_log_tau_device = torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32, device=device))
    else:
        init_log_tau_device = init_log_tau.detach().to(device).float()

    baseline_A = torch.zeros(C, r, dtype=torch.float32, device=device)

    # Global/no-personalization baseline on adapt_val
    baseline_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        cfg,
        device,
    )
    baseline_score = personalization_selection_score(baseline_metrics)

    # Closed-form init from support set
    init_emp, present, _ = extract_confidence_weighted_prototypes(
        frozen_model,
        support_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        num_classes=C,
        emb_dim=d,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    init_A = solve_A_closed_form(
        init_emp.to(device),
        present.to(device),
        global_proto,
        proto_basis,
        cfg.lambda_A,
    ).detach()

    client_A = nn.Parameter(init_A.clone())
    log_tau = nn.Parameter(init_log_tau_device.clone())

    init_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        client_A.detach(),
        log_tau.detach(),
        cfg,
        device,
    )

    hps = choose_personalization_hparams(init_metrics["f1"], cfg)
    adapt_epochs = int(hps["epochs"])
    warm_epochs = int(hps["warm_epochs"])
    anchor_w = float(hps["anchor_w"])
    lr_A = float(hps["lr_A"])
    lr_tau = float(hps["lr_tau"])

    optimizer_A = torch.optim.AdamW(
        [{"params": [client_A], "lr": lr_A, "weight_decay": 1e-4}]
    )
    optimizer_joint = torch.optim.AdamW(
        [
            {"params": [client_A], "lr": lr_A, "weight_decay": 1e-4},
            {"params": [log_tau], "lr": lr_tau, "weight_decay": 0.0},
        ]
    )

    best = {
        "A": client_A.detach().cpu().clone(),
        "log_tau": log_tau.detach().cpu().clone(),
        "score": personalization_selection_score(init_metrics),
        "metrics": init_metrics,
    }

    no_improve = 0
    init_A_device = init_A.detach()

    for epoch in range(adapt_epochs):
        frozen_model.eval()
        optimizer = optimizer_A if epoch < warm_epochs else optimizer_joint

        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            z = frozen_model(xb)
            P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
            logits = proto_logits(
                z,
                global_proto,
                proto_basis,
                client_A,
                log_tau,
                cfg,
            )

            loss_proto = F.cross_entropy(logits, yb, label_smoothing=0.02)
            align = 1.0 - torch.sum(z * P_i[yb], dim=1).mean()
            reg_A = client_A.pow(2).mean()
            anchor_A = (client_A - init_A_device).pow(2).mean()
            anchor_tau = (log_tau - init_log_tau_device).pow(2)

            loss = (
                loss_proto
                + cfg.lambda_align * align
                + cfg.lambda_A * reg_A
                + anchor_w * anchor_A
                + 0.5 * cfg.lambda_tau * anchor_tau
            )

            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                if epoch < warm_epochs:
                    torch.nn.utils.clip_grad_norm_([client_A], cfg.grad_clip)
                else:
                    torch.nn.utils.clip_grad_norm_([client_A, log_tau], cfg.grad_clip)

            optimizer.step()

        val_metrics = evaluate_loader(
            frozen_model,
            adapt_val_loader,
            global_proto,
            proto_basis,
            client_A.detach(),
            log_tau.detach(),
            cfg,
            device,
        )

        score = personalization_selection_score(val_metrics)

        if score > best["score"]:
            best["score"] = score
            best["A"] = client_A.detach().cpu().clone()
            best["log_tau"] = log_tau.detach().cpu().clone()
            best["metrics"] = val_metrics
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.personalization_patience:
            break

    required_margin = cfg.personalization_gate_score_margin
    if baseline_metrics["f1"] >= cfg.easy_subject_f1_threshold:
        required_margin += cfg.personalization_gate_score_margin_easy

    use_personalization = (
        best["score"] > baseline_score + required_margin
        and best["metrics"]["f1"] >= baseline_metrics["f1"] + cfg.personalization_gate_f1_margin
    )

    if use_personalization:
        selected_A = best["A"]
        selected_log_tau = best["log_tau"]
        selected_metrics = best["metrics"]
    else:
        selected_A = baseline_A.detach().cpu().clone()
        selected_log_tau = init_log_tau_device.detach().cpu().clone()
        selected_metrics = baseline_metrics

    gate_info = {
        "used_personalization": bool(use_personalization),
        "baseline_metrics": baseline_metrics,
        "baseline_score": float(baseline_score),
        "personalized_metrics": best["metrics"],
        "personalized_score": float(best["score"]),
        "required_margin": float(required_margin),
    }

    return selected_A, selected_log_tau, selected_metrics, gate_info

## Fold training

Runs one LOSO fold, including federated rounds, validation, global testing, and personalization.

In [ ]:
# ============================================================
# Fold training
# ============================================================


def train_one_fold(
    loaders: Dict[str, object],
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    validate_every = max(1, int(getattr(cfg, "validate_every", 1)))
    show_progress = bool(getattr(cfg, "progress", True))

    global_model = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    proto_basis = orthonormal_random(cfg.emb_dim, cfg.proto_rank, device)

    client_A = {sid: torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32) for sid in train_ids}
    client_log_tau = {
        sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32))
        for sid in train_ids
    }

    best_snapshot = None
    best_score = -1.0
    history: List[Dict[str, float]] = []

    warmup_rounds = min(3, max(1, cfg.rounds // 4))
    round_bar = tqdm(
        range(1, cfg.rounds + 1),
        desc=f"Fold {heldout_id}",
        leave=True,
        disable=(not show_progress),
    )

    for rnd in round_bar:
        local_state_dicts = []
        local_weights = []
        proto_summaries: Dict[str, Dict[str, torch.Tensor]] = {}

        for sid in train_ids:
            state_dict, _A_local, log_tau_local, proto_summary = train_one_client(
                global_model=global_model,
                train_loader=train_clients[sid]["train"],
                val_loader=train_clients[sid]["val"],
                global_proto=global_proto,
                proto_basis=proto_basis,
                A_init=client_A[sid],
                log_tau_init=client_log_tau[sid],
                cfg=cfg,
                device=device,
                warmup_extract=(rnd <= warmup_rounds),
            )

            local_state_dicts.append(state_dict)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries[sid] = proto_summary
            client_log_tau[sid] = log_tau_local.clone()

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        global_proto, proto_basis, new_client_A = server_geometry_update(
            global_proto=global_proto,
            proto_basis=proto_basis,
            proto_summaries=proto_summaries,
            cfg=cfg,
            device=device,
        )
        client_A = new_client_A

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        per_client_val_sizes: Dict[str, int] = {}

        with torch.no_grad():
            for sid in train_ids:
                metrics = evaluate_loader(
                    global_model,
                    train_clients[sid]["val"],
                    global_proto,
                    proto_basis,
                    client_A[sid].to(device),
                    client_log_tau[sid].to(device),
                    cfg,
                    device,
                )
                per_client_metrics[sid] = metrics
                per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(
            per_client_metrics, per_client_val_sizes, cfg
        )

        round_bar.set_postfix({
            "val_f1": f"{mean_f1:.4f}",
            "val_acc": f"{mean_acc:.4f}",
            "val_ece": f"{mean_ece:.4f}",
        })

        history.append({
            "round": rnd,
            "mean_val_f1": mean_f1,
            "mean_val_acc": mean_acc,
            "mean_val_ece": mean_ece,
        })

        if score > best_score:
            best_score = score
            best_snapshot = {
                "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "proto_basis": proto_basis.detach().cpu().clone(),
                "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
                "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
                "history": copy.deepcopy(history),
            }

    if best_snapshot is None:
        best_snapshot = {
            "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
            "global_proto": global_proto.detach().cpu().clone(),
            "proto_basis": proto_basis.detach().cpu().clone(),
            "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
            "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
            "history": copy.deepcopy(history),
        }

    global_model.load_state_dict(best_snapshot["model_state"])
    global_proto = best_snapshot["global_proto"].to(device)
    proto_basis = best_snapshot["proto_basis"].to(device)
    saved_log_tau = best_snapshot["client_log_tau"]

    train_size_weights = torch.tensor(
        [train_clients[sid]["n_train"] for sid in train_ids],
        dtype=torch.float32,
        device=device,
    )
    tau_stack = torch.stack([saved_log_tau[sid].float().to(device) for sid in train_ids], dim=0)
    mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

    global_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        torch.zeros(num_classes, cfg.proto_rank, device=device),
        mean_log_tau,
        cfg,
        device,
    )

    best_A, best_log_tau, adapt_val_metrics, adapt_gate = adapt_heldout_client(
        model=global_model,
        support_loader=heldout["support"],
        adapt_val_loader=heldout["adapt_val"],
        global_proto=global_proto,
        proto_basis=proto_basis,
        cfg=cfg,
        device=device,
        init_log_tau=mean_log_tau.detach(),
    )

    personalized_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        best_A.to(device),
        best_log_tau.to(device),
        cfg,
        device,
    )

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": best_snapshot["history"],
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val_metrics,
        "adapt_gate": adapt_gate,
    }

## Experiment driver

Runs the base LOSO experiment and summarizes global vs personalized results.

In [ ]:
# ============================================================
# Experiment driver
# ============================================================

def pretty_metric_line(name: str, metrics: Dict[str, float]) -> str:
    return (
        f"{name} -> ACC: {metrics['acc']:.4f}, "
        f"Macro-F1: {metrics['f1']:.4f}, "
        f"ECE: {metrics['ece']:.4f}, "
        f"Brier: {metrics['brier']:.4f}"
    )


def run_loso_experiment(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (activity_id -> class_idx): {label_map}")

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        print("Fold summary:")
        for sid, stats in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats.items()])
            print(f"  {sid}: {stats_str}")

        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v)) for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v)) for k, v in pers_metrics.items()}

    print("\n================ FINAL LOSO RESULTS ================")
    print(
        "Global      -> "
        f"ACC: {global_mean['acc']:.4f} ± {global_std['acc']:.4f}, "
        f"Macro-F1: {global_mean['f1']:.4f} ± {global_std['f1']:.4f}, "
        f"ECE: {global_mean['ece']:.4f} ± {global_std['ece']:.4f}, "
        f"Brier: {global_mean['brier']:.4f} ± {global_std['brier']:.4f}"
    )
    print(
        "Personalized-> "
        f"ACC: {pers_mean['acc']:.4f} ± {pers_std['acc']:.4f}, "
        f"Macro-F1: {pers_mean['f1']:.4f} ± {pers_std['f1']:.4f}, "
        f"ECE: {pers_mean['ece']:.4f} ± {pers_std['ece']:.4f}, "
        f"Brier: {pers_mean['brier']:.4f} ± {pers_std['brier']:.4f}"
    )

    gate = result["adapt_gate"]
    print(
        "Adapt gate -> "
        f"used_personalization={gate['used_personalization']} | "
        f"baseline_score={gate['baseline_score']:.4f} | "
        f"personalized_score={gate['personalized_score']:.4f} | "
        f"required_margin={gate['required_margin']:.4f}"
    )

    return {
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }

## Combined benchmark suite section

Starts the benchmark suite that reuses CDPL components and adds baseline methods.

In [ ]:
# ============================================================
# Combined benchmark suite section
# ============================================================
# Reuse aliases from the CCD section above.
CCDConfig = Config
ccd_train_one_fold = train_one_fold

import copy
import json
import math
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Using symbols defined above from the combined base section.

## Benchmark configuration

Benchmark-specific configuration for presets, methods, seeds, fold filtering, saving, and plotting.

In [ ]:
# ============================================================
# Benchmark configuration
# ============================================================


@dataclass
class BenchmarkConfig(Config):
    # preset in {"fast", "screening", "final", "custom"}
    preset: str = "screening"

    # optional subset of methods / folds
    methods: Tuple[str, ...] = ()
    seeds: Tuple[int, ...] = ()
    heldout_ids: Optional[Tuple[str, ...]] = None

    # Default exclusion: subject109 is effectively single-class in this pipeline.
    exclude_subject_ids: Tuple[str, ...] = ("subject109",)
    include_excluded_subjects: bool = False
    min_unique_classes_per_subject: int = 2

    # runtime controls
    validate_every: int = 2
    progress: bool = True
    save_plots: bool = True
    save_histories: bool = True

    # baseline-specific optimization
    head_lr: float = 1e-2
    head_epochs: int = 12
    finetune_epochs: int = 12
    finetune_patience: int = 3

    fedprox_mu: float = 1e-2
    ditto_mu: float = 5e-3
    pfedme_mu: float = 5e-3
    pfedme_beta: float = 0.5
    pfedme_inner_steps: int = 3

    fedrep_head_steps: int = 2
    fedrep_body_steps: int = 1

    fedproto_lambda: float = 0.2
    fedproto_tau: float = 8.0
    fedproto_alpha_grid: Tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)

    benchmark_subdir: str = "baseline_benchmark"
    plot_dpi: int = 180

    def __post_init__(self) -> None:
        super().__post_init__()
        apply_benchmark_preset(self)


FAST_METHODS = (
    "ccd",
    "fedavg",
    "fedprox",
    "fedrep",
    "fedproto",
)

SCREENING_METHODS = FAST_METHODS

FINAL_METHODS = (
    "ccd",
    "fedavg",
    "fedprox",
    "fedrep",
    "fedproto",
    "centralized",
    "localonly",
)

OPTIONAL_METHODS = (
    "ditto",
    "pfedme",
    "fedper",
)


def apply_benchmark_preset(cfg: BenchmarkConfig) -> None:
    preset = (cfg.preset or "custom").lower()

    if preset == "custom":
        if not cfg.methods:
            cfg.methods = FINAL_METHODS
        if not cfg.seeds:
            cfg.seeds = (42,)
        return

    if preset == "fast":
        cfg.methods = FAST_METHODS if not cfg.methods else cfg.methods
        cfg.seeds = (42,) if not cfg.seeds else cfg.seeds
        cfg.rounds = 4
        cfg.local_epochs = 2
        cfg.personalization_epochs_min = 3
        cfg.personalization_epochs_max = 5
        cfg.personalization_patience = 2
        cfg.finetune_epochs = 5
        cfg.server_geom_steps = 6
        cfg.server_alt_iters = 1
        cfg.batch_size = 256
        cfg.personal_batch_size = 128
        cfg.num_workers = 2
        cfg.validate_every = max(1, cfg.validate_every)
        cfg.save_plots = False
    elif preset == "screening":
        cfg.methods = SCREENING_METHODS if not cfg.methods else cfg.methods
        cfg.seeds = (42,) if not cfg.seeds else cfg.seeds
        cfg.rounds = 6
        cfg.local_epochs = 3
        cfg.personalization_epochs_min = 4
        cfg.personalization_epochs_max = 6
        cfg.personalization_patience = 2
        cfg.finetune_epochs = 6
        cfg.server_geom_steps = 8
        cfg.server_alt_iters = 1
        cfg.batch_size = 256
        cfg.personal_batch_size = 128
        cfg.num_workers = 2
        cfg.validate_every = max(1, cfg.validate_every)
        cfg.save_plots = False
    elif preset == "final":
        cfg.methods = FINAL_METHODS if not cfg.methods else cfg.methods
        cfg.seeds = (42, 43, 44) if not cfg.seeds else cfg.seeds
        cfg.rounds = 12
        cfg.local_epochs = 5
        cfg.personalization_epochs_min = 6
        cfg.personalization_epochs_max = 10
        cfg.personalization_patience = 3
        cfg.finetune_epochs = 10
        cfg.server_geom_steps = 12
        cfg.server_alt_iters = 2
        cfg.batch_size = 256
        cfg.personal_batch_size = 128
        cfg.num_workers = 2
        cfg.validate_every = 1
        cfg.save_plots = True
    else:
        raise ValueError(f"Unknown preset: {cfg.preset!r}")

    if not cfg.methods:
        cfg.methods = FINAL_METHODS
    if not cfg.seeds:
        cfg.seeds = (42,)


def should_validate_round(step: int, total_steps: int, validate_every: int) -> bool:
    if step <= 1 or step >= total_steps:
        return True
    validate_every = max(1, int(validate_every))
    return (step % validate_every) == 0

## Generic helpers

Shared utility functions used by benchmark methods and result generation.

In [ ]:
# ============================================================
# Generic helpers
# ============================================================


def benchmark_dir(cfg: BenchmarkConfig) -> Path:
    out = Path(cfg.save_dir) / cfg.benchmark_subdir
    out.mkdir(parents=True, exist_ok=True)
    return out


def score_from_metrics(metrics: Dict[str, float]) -> float:
    return 0.90 * metrics["f1"] + 0.10 * metrics["acc"] - 0.05 * metrics["ece"]


def state_to_device(state: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {k: v.detach().to(device) for k, v in state.items()}


def blend_state_dicts(
    old_state: Dict[str, torch.Tensor],
    new_state: Dict[str, torch.Tensor],
    beta: float,
) -> Dict[str, torch.Tensor]:
    out: Dict[str, torch.Tensor] = {}
    for k in old_state.keys():
        if torch.is_floating_point(old_state[k]):
            out[k] = (1.0 - beta) * old_state[k].float() + beta * new_state[k].float()
            out[k] = out[k].to(dtype=old_state[k].dtype)
        else:
            out[k] = new_state[k].clone()
    return out


def average_selected_state_dicts(
    state_dicts: List[Dict[str, torch.Tensor]],
    weights: List[float],
    prefix: str,
) -> Dict[str, torch.Tensor]:
    total = float(sum(weights))
    weights = [w / total for w in weights]
    out: Dict[str, torch.Tensor] = {}
    keys = [k for k in state_dicts[0].keys() if k.startswith(prefix)]
    for k in keys:
        ref = state_dicts[0][k]
        if torch.is_floating_point(ref):
            acc = None
            for sd, w in zip(state_dicts, weights):
                val = sd[k].float()
                acc = val * w if acc is None else acc + val * w
            out[k] = acc.to(dtype=ref.dtype)
        else:
            out[k] = ref.clone()
    return out


def split_prefixed_state(state: Dict[str, torch.Tensor], prefix: str) -> Dict[str, torch.Tensor]:
    plen = len(prefix)
    return {k[plen:]: v.clone() for k, v in state.items() if k.startswith(prefix)}


def prox_penalty(model: nn.Module, anchor_state: Dict[str, torch.Tensor]) -> torch.Tensor:
    device = next(model.parameters()).device
    penalty = torch.zeros((), dtype=torch.float32, device=device)
    for name, p in model.named_parameters():
        penalty = penalty + (p - anchor_state[name]).pow(2).sum()
    return penalty


def loaders_to_partition(loaders: List[torch.utils.data.DataLoader], seq_len: int, feat_dim: int) -> SubjectPartition:
    xs = []
    ys = []
    for loader in loaders:
        ds = loader.dataset
        if len(ds) == 0:
            continue
        xs.append(ds.x.detach().cpu().numpy())
        ys.append(ds.y.detach().cpu().numpy())
    if not xs:
        return SubjectPartition(
            windows=np.empty((0, seq_len, feat_dim), dtype=np.float32),
            labels=np.empty((0,), dtype=np.int64),
        )
    return SubjectPartition(
        windows=np.concatenate(xs, axis=0).astype(np.float32),
        labels=np.concatenate(ys, axis=0).astype(np.int64),
    )

## Shared classifier baselines

Shared encoder/classifier baselines such as FedAvg/FedRep-style variants.

In [ ]:
# ============================================================
# Shared classifier baselines
# ============================================================

class EncoderClassifier(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, cfg: BenchmarkConfig):
        super().__init__()
        self.encoder = TemporalEncoder(in_dim, cfg)
        self.head = nn.Linear(cfg.emb_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.head(z)


@torch.inference_mode()
def predict_classifier_loader(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    device: torch.device,
    cfg: BenchmarkConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    if not probs_all:
        out_dim = model.head.out_features
        return np.empty((0, out_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_classifier_loader(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    device: torch.device,
    cfg: BenchmarkConfig,
) -> Dict[str, float]:
    probs, labels = predict_classifier_loader(model, loader, device, cfg)
    return summarize_probs(probs, labels, model.head.out_features, cfg.ece_bins)


def make_optimizer(
    model: nn.Module,
    lr: float,
    weight_decay: float,
    freeze_encoder: bool = False,
    freeze_head: bool = False,
) -> torch.optim.Optimizer:
    for p in model.encoder.parameters():
        p.requires_grad = not freeze_encoder
    for p in model.head.parameters():
        p.requires_grad = not freeze_head
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        raise RuntimeError("No trainable parameters selected.")
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)


def train_classifier_epochs(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    cfg: BenchmarkConfig,
    device: torch.device,
    epochs: int,
    lr: Optional[float] = None,
    weight_decay: Optional[float] = None,
    prox_anchor: Optional[Dict[str, torch.Tensor]] = None,
    prox_mu: float = 0.0,
    freeze_encoder: bool = False,
    freeze_head: bool = False,
) -> None:
    model.train()
    lr = cfg.lr_encoder if lr is None else lr
    weight_decay = cfg.weight_decay if weight_decay is None else weight_decay
    optimizer = make_optimizer(model, lr, weight_decay, freeze_encoder, freeze_head)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))
    class_weight = make_class_weight_from_loader(train_loader, model.head.out_features, device, cfg)

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    anchor_dev = state_to_device(prox_anchor, device) if prox_anchor is not None and prox_mu > 0 else None

    for _ in range(epochs):
        epoch_steps = 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.05)
                if anchor_dev is not None and prox_mu > 0:
                    loss = loss + 0.5 * prox_mu * prox_penalty(model, anchor_dev)
            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    epoch_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
                optimizer.step()
                epoch_steps += 1
        if epoch_steps > 0:
            scheduler.step()


def adapt_classifier_model(
    global_model: nn.Module,
    support_loader: torch.utils.data.DataLoader,
    adapt_val_loader: torch.utils.data.DataLoader,
    cfg: BenchmarkConfig,
    device: torch.device,
    mode: str,
    prox_mu: float = 0.0,
) -> Tuple[nn.Module, Dict[str, float]]:
    model = clone_model(global_model).to(device)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_metrics = evaluate_classifier_loader(model, adapt_val_loader, device, cfg)
    best_score = score_from_metrics(best_metrics)
    no_improve = 0
    anchor = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    if mode == "head_only":
        freeze_encoder, freeze_head = True, False
        lr = cfg.head_lr
    else:
        freeze_encoder, freeze_head = False, False
        lr = cfg.lr_personal

    for epoch in range(cfg.finetune_epochs):
        if mode == "pfedme":
            # Moreau-style adaptation: repeated proximal inner steps.
            local_model = model
            local_model.train()
            optimizer = make_optimizer(local_model, cfg.lr_personal, cfg.weight_decay, False, False)
            class_weight = make_class_weight_from_loader(support_loader, local_model.head.out_features, device, cfg)
            anchor_dev = state_to_device(anchor, device)
            for xb, yb in support_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                for _ in range(cfg.pfedme_inner_steps):
                    optimizer.zero_grad(set_to_none=True)
                    logits = local_model(xb)
                    loss = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.02)
                    loss = loss + 0.5 * prox_mu * prox_penalty(local_model, anchor_dev)
                    loss.backward()
                    if cfg.grad_clip and cfg.grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(local_model.parameters(), cfg.grad_clip)
                    optimizer.step()
        else:
            train_classifier_epochs(
                model=model,
                train_loader=support_loader,
                cfg=cfg,
                device=device,
                epochs=1,
                lr=lr,
                weight_decay=cfg.weight_decay,
                prox_anchor=anchor if mode == "ditto" else None,
                prox_mu=prox_mu if mode == "ditto" else 0.0,
                freeze_encoder=freeze_encoder,
                freeze_head=freeze_head,
            )

        metrics = evaluate_classifier_loader(model, adapt_val_loader, device, cfg)
        sc = score_from_metrics(metrics)
        if sc > best_score:
            best_score = sc
            best_metrics = metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= cfg.finetune_patience:
            break

    model.load_state_dict(best_state)
    return model, best_metrics

## Prototype-only baseline (FedProto-style)

FedProto-style baseline implementation using class prototypes.

In [ ]:
# ============================================================
# Prototype-only baseline (FedProto-style)
# ============================================================

@torch.inference_mode()
def predict_proto_loader(
    encoder: nn.Module,
    loader: torch.utils.data.DataLoader,
    proto: torch.Tensor,
    tau: float,
    device: torch.device,
    cfg: BenchmarkConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    encoder.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = encoder(xb)
            logits = float(tau) * (z @ proto.T)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    if not probs_all:
        return np.empty((0, proto.shape[0]), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_proto_loader(
    encoder: nn.Module,
    loader: torch.utils.data.DataLoader,
    proto: torch.Tensor,
    tau: float,
    device: torch.device,
    cfg: BenchmarkConfig,
) -> Dict[str, float]:
    probs, labels = predict_proto_loader(encoder, loader, proto, tau, device, cfg)
    return summarize_probs(probs, labels, proto.shape[0], cfg.ece_bins)


def train_fedproto_local(
    encoder: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    global_proto: torch.Tensor,
    cfg: BenchmarkConfig,
    device: torch.device,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor, torch.Tensor]:
    local_encoder = clone_model(encoder).to(device)
    optimizer = torch.optim.AdamW(local_encoder.parameters(), lr=cfg.lr_encoder, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))
    class_weight = make_class_weight_from_loader(train_loader, global_proto.shape[0], device, cfg)

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    tau = float(cfg.fedproto_tau)
    gp = global_proto.to(device)

    for _ in range(cfg.local_epochs):
        local_encoder.train()
        epoch_steps = 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_encoder(xb)
                logits = tau * (z @ gp.T)
                ce = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.05)
                align = 1.0 - torch.sum(z * gp[yb], dim=1).mean()
                loss = ce + cfg.fedproto_lambda * align
            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(local_encoder.parameters(), cfg.grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    epoch_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(local_encoder.parameters(), cfg.grad_clip)
                optimizer.step()
                epoch_steps += 1
        if epoch_steps > 0:
            scheduler.step()

    empirical, present, counts = extract_confidence_weighted_prototypes(
        model=local_encoder,
        loader=train_loader,
        global_proto=gp,
        proto_basis=torch.zeros(gp.shape[1], 1, dtype=gp.dtype, device=device),
        client_A=torch.zeros(gp.shape[0], 1, dtype=gp.dtype, device=device),
        log_tau=torch.log(torch.tensor(1.0, dtype=torch.float32, device=device)),
        num_classes=gp.shape[0],
        emb_dim=gp.shape[1],
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    return (
        {k: v.detach().cpu().clone() for k, v in local_encoder.state_dict().items()},
        empirical.detach().cpu(),
        present.detach().cpu(),
        counts.detach().cpu(),
    )


def aggregate_fedproto(
    proto_list: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    old_proto: torch.Tensor,
    device: torch.device,
) -> torch.Tensor:
    C, d = old_proto.shape
    numer = torch.zeros(C, d, dtype=torch.float32, device=device)
    denom = torch.zeros(C, dtype=torch.float32, device=device)
    for emp, present, counts in proto_list:
        emp = emp.to(device)
        present = present.to(device)
        counts = counts.to(device).float()
        numer[present] += counts[present].unsqueeze(1) * emp[present]
        denom[present] += counts[present]
    new_proto = old_proto.clone().to(device)
    keep = denom > 0
    new_proto[keep] = numer[keep] / denom[keep].unsqueeze(1)
    new_proto = F.normalize(new_proto, dim=1)
    return new_proto.detach()


def adapt_fedproto_heldout(
    encoder: nn.Module,
    support_loader: torch.utils.data.DataLoader,
    adapt_val_loader: torch.utils.data.DataLoader,
    global_proto: torch.Tensor,
    cfg: BenchmarkConfig,
    device: torch.device,
) -> Tuple[torch.Tensor, Dict[str, float], float]:
    support_emp, support_present, _ = extract_confidence_weighted_prototypes(
        model=encoder,
        loader=support_loader,
        global_proto=global_proto,
        proto_basis=torch.zeros(global_proto.shape[1], 1, dtype=global_proto.dtype, device=device),
        client_A=torch.zeros(global_proto.shape[0], 1, dtype=global_proto.dtype, device=device),
        log_tau=torch.log(torch.tensor(1.0, dtype=torch.float32, device=device)),
        num_classes=global_proto.shape[0],
        emb_dim=global_proto.shape[1],
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    best_alpha = 0.0
    best_metrics = evaluate_proto_loader(encoder, adapt_val_loader, global_proto.to(device), cfg.fedproto_tau, device, cfg)
    best_score = score_from_metrics(best_metrics)
    best_proto = global_proto.detach().cpu().clone()

    for alpha in cfg.fedproto_alpha_grid:
        proto = global_proto.detach().cpu().clone()
        mask = support_present.bool()
        proto[mask] = F.normalize(
            (1.0 - alpha) * proto[mask] + alpha * support_emp[mask],
            dim=1,
        )
        metrics = evaluate_proto_loader(encoder, adapt_val_loader, proto.to(device), cfg.fedproto_tau, device, cfg)
        sc = score_from_metrics(metrics)
        if sc > best_score:
            best_score = sc
            best_metrics = metrics
            best_proto = proto.clone()
            best_alpha = float(alpha)

    return best_proto, best_metrics, best_alpha

## Fold runners

Method-specific fold runners for the benchmark suite.

In [ ]:
# ============================================================
# Fold runners
# ============================================================


def run_ccd_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    return ccd_train_one_fold(loaders, cfg, device)



def run_fedavg_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        local_state_dicts = []
        local_weights = []
        for sid in train_ids:
            local_model = clone_model(global_model).to(device)
            train_classifier_epochs(local_model, train_clients[sid]["train"], cfg, device, cfg.local_epochs)
            local_state_dicts.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
            local_weights.append(train_clients[sid]["n_train"])

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    global_model.load_state_dict(best_state)
    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": None,
        "details": {},
    }

def run_fedprox_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        global_anchor = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
        local_state_dicts = []
        local_weights = []
        for sid in train_ids:
            local_model = clone_model(global_model).to(device)
            train_classifier_epochs(
                local_model,
                train_clients[sid]["train"],
                cfg,
                device,
                cfg.local_epochs,
                prox_anchor=global_anchor,
                prox_mu=cfg.fedprox_mu,
            )
            local_state_dicts.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
            local_weights.append(train_clients[sid]["n_train"])

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    global_model.load_state_dict(best_state)
    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": None,
        "details": {},
    }

def run_fedper_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    local_heads = {sid: copy.deepcopy(global_model.head.state_dict()) for sid in train_ids}
    best_encoder_state = copy.deepcopy(global_model.encoder.state_dict())
    best_global_head = copy.deepcopy(global_model.head.state_dict())
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        local_full_states = []
        local_weights = []
        for sid in train_ids:
            local_model = clone_model(global_model).to(device)
            local_model.head.load_state_dict(local_heads[sid])
            train_classifier_epochs(local_model, train_clients[sid]["train"], cfg, device, cfg.local_epochs)
            state = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}
            local_heads[sid] = copy.deepcopy(local_model.head.state_dict())
            local_full_states.append(state)
            local_weights.append(train_clients[sid]["n_train"])

        enc_avg = average_selected_state_dicts(local_full_states, local_weights, "encoder.")
        head_avg = average_selected_state_dicts(local_full_states, local_weights, "head.")
        global_model.encoder.load_state_dict(split_prefixed_state(enc_avg, "encoder."))
        global_model.head.load_state_dict(split_prefixed_state(head_avg, "head."))

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_encoder_state = copy.deepcopy(global_model.encoder.state_dict())
            best_global_head = copy.deepcopy(global_model.head.state_dict())

    global_model.encoder.load_state_dict(best_encoder_state)
    global_model.head.load_state_dict(best_global_head)

    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    personal_model, adapt_val = adapt_classifier_model(global_model, heldout["support"], heldout["adapt_val"], cfg, device, mode="head_only")
    personalized_test = evaluate_classifier_loader(personal_model, heldout["test"], device, cfg)

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val,
        "details": {"adapt_mode": "head_only"},
    }

def run_fedrep_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    local_heads = {sid: copy.deepcopy(global_model.head.state_dict()) for sid in train_ids}
    best_encoder_state = copy.deepcopy(global_model.encoder.state_dict())
    best_global_head = copy.deepcopy(global_model.head.state_dict())
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        local_full_states = []
        local_weights = []
        for sid in train_ids:
            local_model = clone_model(global_model).to(device)
            local_model.head.load_state_dict(local_heads[sid])
            for _ in range(cfg.local_epochs):
                train_classifier_epochs(
                    local_model,
                    train_clients[sid]["train"],
                    cfg,
                    device,
                    cfg.fedrep_head_steps,
                    lr=cfg.head_lr,
                    freeze_encoder=True,
                    freeze_head=False,
                )
                train_classifier_epochs(
                    local_model,
                    train_clients[sid]["train"],
                    cfg,
                    device,
                    cfg.fedrep_body_steps,
                    lr=cfg.lr_encoder,
                    freeze_encoder=False,
                    freeze_head=True,
                )
            state = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}
            local_heads[sid] = copy.deepcopy(local_model.head.state_dict())
            local_full_states.append(state)
            local_weights.append(train_clients[sid]["n_train"])

        enc_avg = average_selected_state_dicts(local_full_states, local_weights, "encoder.")
        head_avg = average_selected_state_dicts(local_full_states, local_weights, "head.")
        global_model.encoder.load_state_dict(split_prefixed_state(enc_avg, "encoder."))
        global_model.head.load_state_dict(split_prefixed_state(head_avg, "head."))

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_encoder_state = copy.deepcopy(global_model.encoder.state_dict())
            best_global_head = copy.deepcopy(global_model.head.state_dict())

    global_model.encoder.load_state_dict(best_encoder_state)
    global_model.head.load_state_dict(best_global_head)
    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    personal_model, adapt_val = adapt_classifier_model(global_model, heldout["support"], heldout["adapt_val"], cfg, device, mode="head_only")
    personalized_test = evaluate_classifier_loader(personal_model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val,
        "details": {"adapt_mode": "head_only"},
    }

def run_ditto_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    personal_states = {sid: copy.deepcopy(global_model.state_dict()) for sid in train_ids}
    best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        local_state_dicts = []
        local_weights = []
        for sid in train_ids:
            local_model = clone_model(global_model).to(device)
            train_classifier_epochs(local_model, train_clients[sid]["train"], cfg, device, cfg.local_epochs)
            local_state_dicts.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
            local_weights.append(train_clients[sid]["n_train"])
        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        anchor = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
        for sid in train_ids:
            local_personal = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
            local_personal.load_state_dict(personal_states[sid])
            train_classifier_epochs(
                local_personal,
                train_clients[sid]["train"],
                cfg,
                device,
                cfg.local_epochs,
                lr=cfg.lr_personal,
                prox_anchor=anchor,
                prox_mu=cfg.ditto_mu,
            )
            personal_states[sid] = {k: v.detach().cpu().clone() for k, v in local_personal.state_dict().items()}

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    global_model.load_state_dict(best_state)
    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    personal_model, adapt_val = adapt_classifier_model(
        global_model,
        heldout["support"],
        heldout["adapt_val"],
        cfg,
        device,
        mode="ditto",
        prox_mu=cfg.ditto_mu,
    )
    personalized_test = evaluate_classifier_loader(personal_model, heldout["test"], device, cfg)

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val,
        "details": {"adapt_mode": "ditto"},
    }

def run_pfedme_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    personal_states = {sid: copy.deepcopy(global_model.state_dict()) for sid in train_ids}
    best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        anchor = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
        local_state_dicts = []
        local_weights = []
        for sid in train_ids:
            local_personal = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
            local_personal.load_state_dict(personal_states[sid])
            train_classifier_epochs(
                local_personal,
                train_clients[sid]["train"],
                cfg,
                device,
                cfg.local_epochs,
                lr=cfg.lr_personal,
                prox_anchor=anchor,
                prox_mu=cfg.pfedme_mu,
            )
            personal_states[sid] = {k: v.detach().cpu().clone() for k, v in local_personal.state_dict().items()}
            local_state_dicts.append(personal_states[sid])
            local_weights.append(train_clients[sid]["n_train"])

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_state_cpu = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}
        blended = blend_state_dicts(global_state_cpu, avg_state, cfg.pfedme_beta)
        global_model.load_state_dict(blended)

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_classifier_loader(global_model, train_clients[sid]["val"], device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    global_model.load_state_dict(best_state)
    global_test = evaluate_classifier_loader(global_model, heldout["test"], device, cfg)
    personal_model, adapt_val = adapt_classifier_model(
        global_model,
        heldout["support"],
        heldout["adapt_val"],
        cfg,
        device,
        mode="pfedme",
        prox_mu=cfg.pfedme_mu,
    )
    personalized_test = evaluate_classifier_loader(personal_model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val,
        "details": {"adapt_mode": "pfedme"},
    }

def run_centralized_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    train_partition = loaders_to_partition([train_clients[sid]["train"] for sid in train_ids], cfg.seq_len, feature_dim)
    val_partition = loaders_to_partition([train_clients[sid]["val"] for sid in train_ids], cfg.seq_len, feature_dim)
    train_loader = make_loader(train_partition, cfg.batch_size, True, cfg)
    val_loader = make_loader(val_partition, cfg.batch_size, False, cfg)

    model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_metrics = evaluate_classifier_loader(model, val_loader, device, cfg)
    best_score = score_from_metrics(best_metrics)
    history: List[Dict[str, float]] = []

    total_epochs = max(1, cfg.rounds * cfg.local_epochs)
    for epoch in range(1, total_epochs + 1):
        train_classifier_epochs(model, train_loader, cfg, device, 1)
        if not should_validate_round(epoch, total_epochs, validate_every):
            continue
        val_metrics = evaluate_classifier_loader(model, val_loader, device, cfg)
        sc = score_from_metrics(val_metrics)
        history.append({"round": epoch, "mean_val_f1": val_metrics["f1"], "mean_val_acc": val_metrics["acc"], "mean_val_ece": val_metrics["ece"]})
        if sc > best_score:
            best_score = sc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    global_test = evaluate_classifier_loader(model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": None,
        "details": {},
    }

def run_localonly_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    model = EncoderClassifier(feature_dim, num_classes, cfg).to(device)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_metrics = evaluate_classifier_loader(model, heldout["adapt_val"], device, cfg)
    best_score = score_from_metrics(best_metrics)
    history: List[Dict[str, float]] = []

    total_epochs = max(1, cfg.finetune_epochs)
    for epoch in range(1, total_epochs + 1):
        train_classifier_epochs(model, heldout["support"], cfg, device, 1, lr=cfg.lr_personal)
        if not should_validate_round(epoch, total_epochs, validate_every):
            continue
        val_metrics = evaluate_classifier_loader(model, heldout["adapt_val"], device, cfg)
        sc = score_from_metrics(val_metrics)
        history.append({"round": epoch, "mean_val_f1": val_metrics["f1"], "mean_val_acc": val_metrics["acc"], "mean_val_ece": val_metrics["ece"]})
        if sc > best_score:
            best_score = sc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    personal_test = evaluate_classifier_loader(model, heldout["test"], device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": None,
        "personalized_test": personal_test,
        "details": {},
    }

def run_fedproto_fold(loaders: Dict[str, object], cfg: BenchmarkConfig, device: torch.device) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]
    heldout_id = loaders["heldout_id"]
    validate_every = max(1, cfg.validate_every)

    global_encoder = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    best_state = {k: v.detach().cpu().clone() for k, v in global_encoder.state_dict().items()}
    best_proto = global_proto.detach().cpu().clone()
    best_score = -1.0
    history: List[Dict[str, float]] = []

    for rnd in range(1, cfg.rounds + 1):
        local_state_dicts = []
        local_weights = []
        proto_summaries = []
        for sid in train_ids:
            state, emp, present, counts = train_fedproto_local(global_encoder, train_clients[sid]["train"], global_proto, cfg, device)
            local_state_dicts.append(state)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries.append((emp, present, counts))

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_encoder.load_state_dict(avg_state)
        global_proto = aggregate_fedproto(proto_summaries, global_proto, device)

        if not should_validate_round(rnd, cfg.rounds, validate_every):
            continue

        per_client_metrics = {}
        per_client_val_sizes = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_proto_loader(global_encoder, train_clients[sid]["val"], global_proto, cfg.fedproto_tau, device, cfg)
            per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in global_encoder.state_dict().items()}
            best_proto = global_proto.detach().cpu().clone()

    global_encoder.load_state_dict(best_state)
    global_proto = best_proto.to(device)
    global_test = evaluate_proto_loader(global_encoder, heldout["test"], global_proto, cfg.fedproto_tau, device, cfg)
    heldout_proto, adapt_val, best_alpha = adapt_fedproto_heldout(global_encoder, heldout["support"], heldout["adapt_val"], global_proto, cfg, device)
    personalized_test = evaluate_proto_loader(global_encoder, heldout["test"], heldout_proto.to(device), cfg.fedproto_tau, device, cfg)
    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": history,
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val,
        "details": {"best_alpha": best_alpha},
    }

METHOD_RUNNERS = {
    "ccd": run_ccd_fold,
    "fedavg": run_fedavg_fold,
    "fedprox": run_fedprox_fold,
    "ditto": run_ditto_fold,
    "pfedme": run_pfedme_fold,
    "fedper": run_fedper_fold,
    "fedrep": run_fedrep_fold,
    "fedproto": run_fedproto_fold,
    "centralized": run_centralized_fold,
    "localonly": run_localonly_fold,
}

## Plotting and saving

Saves plots, histories, per-fold results, confidence intervals, and statistical summaries.

In [ ]:
# ============================================================
# Plotting and saving
# ============================================================


def save_history_csv(result: Dict[str, object], method: str, seed: int, out_dir: Path) -> None:
    hist = result.get("history", None)
    if not hist:
        return
    df = pd.DataFrame(hist)
    df.to_csv(out_dir / f"history_{method}_seed{seed}_{result['heldout_id']}.csv", index=False)


def plot_bar_summary(df: pd.DataFrame, out_dir: Path, metric: str, mode: str, dpi: int) -> None:
    sub = df[df["mode"] == mode].copy()
    if sub.empty:
        return
    agg = sub.groupby("method")[metric].agg(["mean", "std"]).sort_values("mean", ascending=(metric in {"ece", "brier"}))
    plt.figure(figsize=(11, 5))
    x = np.arange(len(agg))
    plt.bar(x, agg["mean"].values, yerr=agg["std"].values, capsize=4)
    plt.xticks(x, agg.index.tolist(), rotation=30, ha="right")
    plt.ylabel(metric.upper())
    plt.title(f"{mode.title()} {metric.upper()} across methods")
    plt.tight_layout()
    plt.savefig(out_dir / f"bar_{mode}_{metric}.png", dpi=dpi)
    plt.close()


def plot_box_by_method(df: pd.DataFrame, out_dir: Path, metric: str, mode: str, dpi: int) -> None:
    sub = df[df["mode"] == mode].copy()
    if sub.empty:
        return
    methods = sorted(sub["method"].unique())
    data = [sub[sub["method"] == m][metric].values for m in methods]
    plt.figure(figsize=(12, 5))
    plt.boxplot(data, labels=methods, showmeans=True)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric.upper())
    plt.title(f"Per-fold {metric.upper()} distribution ({mode})")
    plt.tight_layout()
    plt.savefig(out_dir / f"box_{mode}_{metric}.png", dpi=dpi)
    plt.close()


def plot_subject_heatmap(df: pd.DataFrame, out_dir: Path, metric: str, mode: str, dpi: int) -> None:
    sub = df[df["mode"] == mode].copy()
    if sub.empty:
        return
    pivot = sub.pivot_table(index="method", columns="heldout_id", values=metric, aggfunc="mean")
    plt.figure(figsize=(12, 6))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label=metric.upper())
    plt.xticks(np.arange(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
    plt.yticks(np.arange(len(pivot.index)), pivot.index)
    plt.title(f"Mean {metric.upper()} by method and held-out subject ({mode})")
    plt.tight_layout()
    plt.savefig(out_dir / f"heatmap_{mode}_{metric}.png", dpi=dpi)
    plt.close()


def plot_personalization_gain(df: pd.DataFrame, out_dir: Path, dpi: int) -> None:
    global_df = df[df["mode"] == "global"][ ["method", "seed", "heldout_id", "f1"] ].rename(columns={"f1": "global_f1"})
    personal_df = df[df["mode"] == "personalized"][ ["method", "seed", "heldout_id", "f1"] ].rename(columns={"f1": "personalized_f1"})
    merged = global_df.merge(personal_df, on=["method", "seed", "heldout_id"], how="inner")
    if merged.empty:
        return
    merged["gain"] = merged["personalized_f1"] - merged["global_f1"]
    methods = sorted(merged["method"].unique())
    data = [merged[merged["method"] == m]["gain"].values for m in methods]
    plt.figure(figsize=(10, 5))
    plt.boxplot(data, labels=methods, showmeans=True)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Macro-F1 gain")
    plt.title("Personalization gain over global model")
    plt.tight_layout()
    plt.savefig(out_dir / "box_personalization_gain_f1.png", dpi=dpi)
    plt.close()


def save_summary_tables(df: pd.DataFrame, out_dir: Path) -> None:
    agg = df.groupby(["method", "mode"])[["acc", "f1", "ece", "brier", "runtime_sec"]].agg(["mean", "std"])
    agg.to_csv(out_dir / "summary_mean_std.csv")

    compact = (
        df.groupby(["method", "mode"])[["acc", "f1", "ece", "brier"]]
        .agg(["mean", "std"])
        .reset_index()
    )
    compact.to_csv(out_dir / "summary_compact.csv", index=False)


def write_manifest(cfg: BenchmarkConfig, out_dir: Path) -> None:
    with open(out_dir / "benchmark_config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

## Benchmark driver

Main benchmark loop for running selected methods and folds.

In [ ]:
# ============================================================
# Benchmark driver
# ============================================================



def build_subject_arrays(cfg: BenchmarkConfig) -> Tuple[List[str], Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[int, int]]:
    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    raw_subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    excluded = set()
    if not cfg.include_excluded_subjects:
        excluded |= set(cfg.exclude_subject_ids)

    for sid, (_X, y) in raw_subject_arrays.items():
        if sid in excluded:
            continue
        unique_classes = int(np.unique(y).size)
        if unique_classes < max(1, int(cfg.min_unique_classes_per_subject)):
            excluded.add(sid)

    subject_arrays = {sid: arr for sid, arr in raw_subject_arrays.items() if sid not in excluded}
    subject_ids = sorted(subject_arrays.keys())

    if excluded:
        print(f"[INFO] Excluding subjects from benchmark: {sorted(excluded)}")

    if cfg.heldout_ids is not None:
        missing = sorted(set(cfg.heldout_ids) - set(subject_ids))
        if missing:
            raise ValueError(f"Unknown or excluded heldout_ids={missing}. Available: {subject_ids}")
        allowed = set(cfg.heldout_ids)
        subject_ids = [sid for sid in subject_ids if sid in allowed]

    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    return subject_ids, subject_arrays, label_map

## Review-1 statistical benchmark output only

Generates the reviewer-required per-subject CSV, 95% confidence intervals, and paired t-test outputs.

In [ ]:
# ============================================================
# Review-1 statistical benchmark output only
# ============================================================
# This section replaces the generic benchmark-output layer.
# It keeps the attached training/evaluation implementations unchanged, but saves
# only the reviewer-required statistical files for seed 42 and subjects101-108.

from scipy import stats

REVIEW1_SUBJECTS: Tuple[str, ...] = tuple(f"subject{i}" for i in range(101, 109))
REVIEW1_METHODS: Tuple[str, ...] = ("ccd", "fedavg", "fedproto", "fedrep")

# method_key -> (paper method name, required variant, result key to read)
REVIEW1_OUTPUT_MAP: Dict[str, Tuple[str, str, str]] = {
    "ccd": ("CDPL", "Personalized", "personalized_test"),
    "fedavg": ("FedAvg", "Global", "global_test"),
    "fedproto": ("FedProto", "Personalized", "personalized_test"),
    "fedrep": ("FedRep", "Personalized", "personalized_test"),
}

REVIEW1_METRICS: Tuple[str, ...] = ("accuracy", "macro_f1", "ece", "brier")
LOWER_IS_BETTER: Tuple[str, ...] = ("ece", "brier")


def review1_benchmark_dir(cfg: BenchmarkConfig) -> Path:
    out = Path(cfg.save_dir) / cfg.benchmark_subdir
    out.mkdir(parents=True, exist_ok=True)
    return out


def mean_ci(values: np.ndarray, confidence: float = 0.95) -> Tuple[float, float, float, float, int]:
    """Return mean, std(ddof=1), CI-low, CI-high, and n using Student's t interval."""
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    n = int(arr.size)
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, 0
    mean = float(arr.mean())
    if n == 1:
        return mean, 0.0, mean, mean, n
    sd = float(arr.std(ddof=1))
    se = sd / math.sqrt(n)
    tcrit = float(stats.t.ppf((1.0 + confidence) / 2.0, n - 1))
    return mean, sd, mean - tcrit * se, mean + tcrit * se, n


def collect_review1_row(method_key: str, seed: int, result: Dict[str, object]) -> Dict[str, object]:
    """Collect exactly one reviewer-required row for a method/fold result."""
    paper_method, variant, result_key = REVIEW1_OUTPUT_MAP[method_key]
    metrics = result.get(result_key)
    if metrics is None:
        raise RuntimeError(
            f"Required result {result_key!r} is missing for method {method_key!r} "
            f"on held-out subject {result.get('heldout_id')!r}."
        )
    return {
        "subject": result["heldout_id"],
        "method": paper_method,
        "variant": variant,
        "accuracy": float(metrics["acc"]),
        "macro_f1": float(metrics["f1"]),
        "ece": float(metrics["ece"]),
        "brier": float(metrics["brier"]),
    }


def save_review1_confidence_intervals(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    method_order = [REVIEW1_OUTPUT_MAP[m][0] for m in REVIEW1_METHODS]

    for method in method_order:
        method_df = df[df["method"] == method].copy()
        if method_df.empty:
            continue
        variant = str(method_df["variant"].iloc[0])
        row: Dict[str, object] = {
            "method": method,
            "variant": variant,
            "n": int(method_df["subject"].nunique()),
        }
        for metric in REVIEW1_METRICS:
            mean, sd, low, high, n = mean_ci(method_df[metric].to_numpy(dtype=float))
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = sd
            row[f"{metric}_ci95_low"] = low
            row[f"{metric}_ci95_high"] = high
            row[f"{metric}_ci95"] = f"{mean:.4f} [{low:.4f}, {high:.4f}]"
            row[f"{metric}_n"] = n
        rows.append(row)

    ci_df = pd.DataFrame(rows)
    ci_df.to_csv(out_dir / "loso_95ci_results.csv", index=False)
    return ci_df


def save_review1_paired_ttests(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    cdpl = df[(df["method"] == "CDPL") & (df["variant"] == "Personalized")].copy()
    if cdpl.empty:
        raise RuntimeError("CDPL Personalized rows are missing; cannot compute paired t-tests.")

    comparisons: Tuple[Tuple[str, str], ...] = (
        ("FedAvg", "Global"),
        ("FedProto", "Personalized"),
        ("FedRep", "Personalized"),
    )

    rows: List[Dict[str, object]] = []
    for baseline_method, baseline_variant in comparisons:
        base = df[(df["method"] == baseline_method) & (df["variant"] == baseline_variant)].copy()
        if base.empty:
            raise RuntimeError(f"Missing rows for {baseline_method} {baseline_variant}.")

        merged = cdpl.merge(base, on="subject", suffixes=("_cdpl", "_base"), how="inner")
        if len(merged) != len(REVIEW1_SUBJECTS):
            raise RuntimeError(
                f"Expected {len(REVIEW1_SUBJECTS)} matched folds for CDPL vs {baseline_method}, "
                f"but found {len(merged)}."
            )

        for metric in REVIEW1_METRICS:
            cdpl_values = merged[f"{metric}_cdpl"].to_numpy(dtype=np.float64)
            base_values = merged[f"{metric}_base"].to_numpy(dtype=np.float64)

            if metric in LOWER_IS_BETTER:
                diff = base_values - cdpl_values
                metric_label = f"{metric}_reduction"
                direction = f"{baseline_method} - CDPL; positive means lower {metric} for CDPL"
            else:
                diff = cdpl_values - base_values
                metric_label = metric
                direction = f"CDPL - {baseline_method}; positive means higher {metric} for CDPL"

            mean_diff, sd_diff, low, high, n = mean_ci(diff)
            t_res = stats.ttest_1samp(diff, popmean=0.0, nan_policy="omit")
            t_stat = float(t_res.statistic) if np.isfinite(t_res.statistic) else np.nan
            p_value = float(t_res.pvalue) if np.isfinite(t_res.pvalue) else np.nan

            rows.append({
                "comparison": f"CDPL vs {baseline_method}",
                "baseline_method": baseline_method,
                "baseline_variant": baseline_variant,
                "metric": metric_label,
                "n": n,
                "mean_paired_difference": mean_diff,
                "std_paired_difference": sd_diff,
                "ci95_low": low,
                "ci95_high": high,
                "ci95": f"[{low:.4f}, {high:.4f}]",
                "t_statistic": t_stat,
                "p_value": p_value,
                "significant_p_0_05": bool(p_value < 0.05) if np.isfinite(p_value) else False,
                "direction_definition": direction,
            })

    stat_df = pd.DataFrame(rows)
    stat_df.to_csv(out_dir / "cdpl_paired_ttest_results.csv", index=False)
    return stat_df


def validate_review1_outputs(df: pd.DataFrame) -> None:
    expected_subjects = set(REVIEW1_SUBJECTS)
    expected_pairs = {(REVIEW1_OUTPUT_MAP[m][0], REVIEW1_OUTPUT_MAP[m][1]) for m in REVIEW1_METHODS}

    actual_subjects = set(df["subject"].unique())
    if actual_subjects != expected_subjects:
        raise RuntimeError(
            f"Subject mismatch. Expected {sorted(expected_subjects)}, found {sorted(actual_subjects)}."
        )

    actual_pairs = set(zip(df["method"], df["variant"]))
    if actual_pairs != expected_pairs:
        raise RuntimeError(
            f"Method/variant mismatch. Expected {sorted(expected_pairs)}, found {sorted(actual_pairs)}."
        )

    expected_rows = len(REVIEW1_SUBJECTS) * len(REVIEW1_METHODS)
    if len(df) != expected_rows:
        raise RuntimeError(f"Expected {expected_rows} rows, found {len(df)} rows.")

    duplicated = df.duplicated(subset=["subject", "method", "variant"]).sum()
    if duplicated:
        raise RuntimeError(f"Found {duplicated} duplicated subject-method-variant rows.")

    if df[list(REVIEW1_METRICS)].isna().any().any():
        raise RuntimeError("At least one required metric is NaN.")


def run_review1_statistical_benchmark(cfg: BenchmarkConfig) -> Dict[str, object]:
    """
    Train/evaluate only the reviewer-required seed-42 baseline benchmark.

    Saved files:
      1. baseline_per_subject_results.csv
      2. loso_95ci_results.csv
      3. cdpl_paired_ttest_results.csv
    """
    cfg.seed = 42
    cfg.seeds = (42,)
    cfg.methods = REVIEW1_METHODS
    cfg.heldout_ids = REVIEW1_SUBJECTS
    cfg.exclude_subject_ids = ("subject109",)
    cfg.include_excluded_subjects = False
    cfg.save_plots = False
    cfg.save_histories = False

    out_dir = review1_benchmark_dir(cfg)
    subject_ids, subject_arrays, label_map = build_subject_arrays(cfg)
    device = torch.device(cfg.device)

    if tuple(subject_ids) != REVIEW1_SUBJECTS:
        raise RuntimeError(f"Expected held-out subjects {REVIEW1_SUBJECTS}, but got {tuple(subject_ids)}.")

    print("\n================ REVIEW-1 STATISTICAL BENCHMARK ================")
    print("Seed: 42")
    print(f"Methods: {cfg.methods}")
    print(f"Subjects: {subject_ids}")
    print("Excluded from baseline comparison: subject109")
    print(f"Output directory: {out_dir}")
    print(f"Label map: {label_map}")

    rows: List[Dict[str, object]] = []
    set_seed(42)

    subject_bar = tqdm(subject_ids, desc="Seed 42 folds", leave=True, disable=(not cfg.progress))
    for heldout_id in subject_bar:
        print(f"\n===== HELD-OUT {heldout_id} =====")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        for method_key in cfg.methods:
            print(f"\n--- Training/evaluating {REVIEW1_OUTPUT_MAP[method_key][0]} on {heldout_id} ---")
            runner = METHOD_RUNNERS[method_key]
            result = runner(loaders, cfg, device)
            rows.append(collect_review1_row(method_key, 42, result))

            required_result_key = REVIEW1_OUTPUT_MAP[method_key][2]
            required_metrics = result[required_result_key]
            print(pretty_metric_line(
                f"{REVIEW1_OUTPUT_MAP[method_key][0]} {REVIEW1_OUTPUT_MAP[method_key][1]}",
                required_metrics,
            ))

            del result
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del fold, loaders
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows, columns=["subject", "method", "variant", "accuracy", "macro_f1", "ece", "brier"])
    df["subject"] = pd.Categorical(df["subject"], categories=list(REVIEW1_SUBJECTS), ordered=True)
    df["method"] = pd.Categorical(
        df["method"],
        categories=[REVIEW1_OUTPUT_MAP[m][0] for m in REVIEW1_METHODS],
        ordered=True,
    )
    df = df.sort_values(["subject", "method"]).reset_index(drop=True)
    df["subject"] = df["subject"].astype(str)
    df["method"] = df["method"].astype(str)

    validate_review1_outputs(df)

    per_subject_path = out_dir / "baseline_per_subject_results.csv"
    df.to_csv(per_subject_path, index=False)
    ci_df = save_review1_confidence_intervals(df, out_dir)
    ttest_df = save_review1_paired_ttests(df, out_dir)

    print("\n================ SAVED REVIEW-1 FILES ================")
    print(per_subject_path)
    print(out_dir / "loso_95ci_results.csv")
    print(out_dir / "cdpl_paired_ttest_results.csv")

    print("\n================ 95% CI TABLE ================")
    print(ci_df.to_string(index=False))
    print("\n================ PAIRED T-TEST TABLE ================")
    print(ttest_df.to_string(index=False))

    return {
        "baseline_per_subject_results": df,
        "ci_results": ci_df,
        "paired_ttest_results": ttest_df,
        "label_map": label_map,
        "output_dir": str(out_dir),
    }



# Manual launcher for the full Review-1 statistical benchmark.
# Set RUN_FULL_REVIEW1_BENCHMARK = True only when you are ready to run the full training/evaluation.
RUN_FULL_REVIEW1_BENCHMARK = False

if RUN_FULL_REVIEW1_BENCHMARK:
    cfg = BenchmarkConfig(
        preset="custom",
        data_dir="/content/drive/MyDrive/PAMAP2_Dataset/Protocol",
        save_dir="/content/drive/MyDrive/baseline1",
        benchmark_subdir="review1_stats_seed42_subject101_108",
        methods=REVIEW1_METHODS,
        seeds=(42,),
        heldout_ids=REVIEW1_SUBJECTS,
        exclude_subject_ids=("subject109",),
        include_excluded_subjects=False,
        progress=True,
        save_plots=False,
        save_histories=False,

        # Paper/config defaults retained unless explicitly changed here.
        # These values match the attached CDPL training setup rather than the older multi-seed preset.
        rounds=15,
        local_epochs=6,
        personalization_epochs=10,
        batch_size=128,
        personal_batch_size=64,
        num_workers=0,
    )

    run_review1_statistical_benchmark(cfg)
else:
    print("Full Review-1 benchmark is disabled. Set RUN_FULL_REVIEW1_BENCHMARK = True to run it.")